In [ ]:
"""
Drug Target Detection Pipeline
================================
Identifies protein targets for a given drug using:
  1. PubMed abstract retrieval for drug context
  2. LLM-based drug description generation
  3. LLM-based description refinement
  4. K-shot LLM inference over a candidate protein list (self-consistency)

Dependencies:
    pip install anthropic biopython requests tqdm
"""

import json
import time
from collections import defaultdict
from typing import Optional

import anthropic
import requests
from Bio import Entrez
from tqdm import tqdm


# ---------------------------------------------------------------------------
# 1. Abstract Retrieval (PubMed)
# ---------------------------------------------------------------------------

def fetch_abstracts(drug_name: str, k: int = 10, mail: Optional[str] = None) -> list[str]:
    """
    Fetch up to k PubMed abstracts most relevant to a drug.

    Args:
        drug_name: Name of the drug to query.
        k:         Maximum number of abstracts to retrieve.
        mail:      Email for NCBI Entrez (recommended to avoid rate limits).

    Returns:
        List of abstract strings.
    """
    Entrez.email = mail or "anonymous@example.com"

    handle = Entrez.esearch(db="pubmed", term=drug_name, retmax=k, sort="relevance")
    search_results = Entrez.read(handle)
    handle.close()
    id_list = search_results["IdList"]

    if not id_list:
        print(f"[fetch_abstracts] No PubMed results found for '{drug_name}'.")
        return []

    handle = Entrez.efetch(db="pubmed", id=id_list, rettype="xml")
    papers = Entrez.read(handle)["PubmedArticle"]
    handle.close()

    abstracts = []
    for i, paper in enumerate(papers):
        try:
            text = paper["MedlineCitation"]["Article"]["Abstract"]["AbstractText"][0]
            abstracts.append(str(text))
        except KeyError:
            print(f"[fetch_abstracts] Abstract {i + 1} not available — skipped.")

    return abstracts


# ---------------------------------------------------------------------------
# 2. STRING Protein List Retrieval
# ---------------------------------------------------------------------------

def fetch_string_proteins(
    gene_or_protein: str,
    species: int = 9606,
    limit: int = 50,
    score_threshold: int = 700,
) -> list[str]:
    """
    Retrieve a list of interacting proteins from the STRING database.

    This returns the neighborhood of `gene_or_protein` in the STRING
    interaction network, filtered by a confidence score threshold.  The
    resulting list is used as the candidate target space for the LLM.

    Args:
        gene_or_protein:  Gene symbol or protein name (e.g. "TUBB", "TP53").
        species:          NCBI taxonomy ID (default 9606 = Homo sapiens).
        limit:            Maximum number of interactors to return.
        score_threshold:  Minimum STRING combined score (0–1000).

    Returns:
        Sorted list of unique gene/protein names.
    """
    url = "https://string-db.org/api/json/network"
    params = {
        "identifiers": gene_or_protein,
        "species": species,
        "limit": limit,
        "caller_identity": "drug_target_pipeline",
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except Exception as exc:
        print(f"[fetch_string_proteins] STRING API error: {exc}")
        return []

    proteins: set[str] = set()
    for edge in data:
        if edge.get("score", 0) >= score_threshold / 1000:
            proteins.add(edge["preferredName_A"])
            proteins.add(edge["preferredName_B"])

    return sorted(proteins)


# ---------------------------------------------------------------------------
# 3. Stage 1 — Drug Description Generation
# ---------------------------------------------------------------------------

DESCRIPTION_SYSTEM_PROMPT = """You are a biomedical expert specialising in pharmacology and 
drug mechanisms of action. Your task is to synthesise information about a drug from 
multiple sources and produce a structured, scientifically accurate description."""

DESCRIPTION_USER_TEMPLATE = """Using the information below, write a comprehensive description 
of the drug including:
- Primary mechanism of action
- Known molecular targets
- Key biological pathways involved
- Pharmacokinetics (metabolism, excretion)
- Notable clinical applications and effects

=== DRUG INFORMATION ===
Name:      {drug_name}
SMILES:    {smiles}
InChI Key: {inchi_key}

=== RELEVANT PUBMED ABSTRACTS ===
{abstracts}

Write a structured, detailed description. Focus on molecular biology and pharmacology."""


def generate_drug_description(
    drug_name: str,
    smiles: str,
    inchi_key: str,
    abstracts: list[str],
    client: anthropic.Anthropic,
    model: str = "claude-sonnet-4-20250514",
) -> str:
    """
    Stage 1: Ask the LLM to produce a preliminary drug description.

    Args:
        drug_name:  Name of the drug.
        smiles:     SMILES string.
        inchi_key:  InChI Key.
        abstracts:  PubMed abstracts retrieved earlier.
        client:     Anthropic client instance.
        model:      Model identifier.

    Returns:
        LLM-generated description string.
    """
    abstracts_text = "\n\n".join(
        f"[Abstract {i + 1}]\n{a}" for i, a in enumerate(abstracts)
    )
    user_prompt = DESCRIPTION_USER_TEMPLATE.format(
        drug_name=drug_name,
        smiles=smiles,
        inchi_key=inchi_key,
        abstracts=abstracts_text,
    )

    message = client.messages.create(
        model=model,
        max_tokens=1500,
        system=DESCRIPTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return message.content[0].text


# ---------------------------------------------------------------------------
# 4. Stage 2 — Description Refinement
# ---------------------------------------------------------------------------

REFINE_SYSTEM_PROMPT = """You are a scientific editor specialising in drug biology. 
Your task is to refine and condense a drug description, removing redundancy while 
preserving all mechanistic detail. The output will be used downstream to identify 
molecular protein targets."""

REFINE_USER_TEMPLATE = """Refine the following drug description. 
Keep it concise (300–500 words). Emphasise:
- Core mechanism at the molecular level
- Direct protein/enzyme interactions
- Downstream signalling effects

=== DRUG DESCRIPTION ===
{description}

Output only the refined description, no preamble."""


def refine_drug_description(
    description: str,
    client: anthropic.Anthropic,
    model: str = "claude-sonnet-4-20250514",
) -> str:
    """
    Stage 2: Refine the preliminary description to a concise, target-focused text.

    Args:
        description: Raw description from Stage 1.
        client:      Anthropic client instance.
        model:       Model identifier.

    Returns:
        Refined description string.
    """
    user_prompt = REFINE_USER_TEMPLATE.format(description=description)
    message = client.messages.create(
        model=model,
        max_tokens=800,
        system=REFINE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return message.content[0].text


# ---------------------------------------------------------------------------
# 5. Stage 3 — Target Detection (Few-Shot, Self-Consistency)
# ---------------------------------------------------------------------------

TARGET_SYSTEM_PROMPT = """You are a computational biologist specialising in drug-target 
interaction prediction. Given a drug description and a list of candidate proteins, 
select the most likely molecular targets of the drug.

Rules:
- You MUST select targets only from the provided candidate protein list.
- Return your answer as valid JSON with keys "targets" (list of protein names) 
  and "rationale" (one sentence per target).
- Select between 1 and {max_targets} targets.
- Be conservative: only include targets you are confident about."""

TARGET_USER_TEMPLATE = """=== FEW-SHOT EXAMPLES ===
{few_shot_block}

=== TASK ===
Drug Description:
{refined_description}

Candidate Proteins (select only from this list):
{protein_list}

Return JSON only, no extra text:
{{
  "targets": ["ProteinA", "ProteinB"],
  "rationale": {{
    "ProteinA": "one-sentence rationale",
    "ProteinB": "one-sentence rationale"
  }}
}}"""

FEW_SHOT_EXAMPLE_TEMPLATE = """--- Example {idx} ---
Drug: {drug_name}
Description: {description}
Selected Targets: {targets}
"""


def _build_few_shot_block(few_shot_examples: list[dict]) -> str:
    """Format few-shot examples into a single text block."""
    blocks = []
    for i, ex in enumerate(few_shot_examples, 1):
        blocks.append(
            FEW_SHOT_EXAMPLE_TEMPLATE.format(
                idx=i,
                drug_name=ex["drug_name"],
                description=ex["description"],
                targets=", ".join(ex["targets"]),
            )
        )
    return "\n".join(blocks) if blocks else "No examples provided."


def _parse_target_response(text: str) -> dict:
    """Robustly parse a JSON response from the target-detection LLM."""
    # Strip markdown fences if present
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # Fallback: try to extract JSON object
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end > start:
            try:
                return json.loads(text[start:end])
            except json.JSONDecodeError:
                pass
    return {"targets": [], "rationale": {}}


def detect_targets_single_run(
    refined_description: str,
    protein_list: list[str],
    few_shot_examples: list[dict],
    client: anthropic.Anthropic,
    model: str = "claude-sonnet-4-20250514",
    max_targets: int = 5,
    temperature: float = 0.7,
) -> dict:
    """
    Run a single LLM inference to predict targets.

    Args:
        refined_description: Refined drug description from Stage 2.
        protein_list:        Candidate proteins (e.g. from STRING).
        few_shot_examples:   List of dicts with keys: drug_name, description, targets.
        client:              Anthropic client.
        model:               Model identifier.
        max_targets:         Maximum targets the model may select.
        temperature:         Sampling temperature (higher → more diverse).

    Returns:
        Dict with 'targets' (list) and 'rationale' (dict).
    """
    few_shot_block = _build_few_shot_block(few_shot_examples)
    protein_list_str = "\n".join(f"  - {p}" for p in protein_list)

    user_prompt = TARGET_USER_TEMPLATE.format(
        few_shot_block=few_shot_block,
        refined_description=refined_description,
        protein_list=protein_list_str,
    )
    system = TARGET_SYSTEM_PROMPT.format(max_targets=max_targets)

    message = client.messages.create(
        model=model,
        max_tokens=600,
        system=system,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return _parse_target_response(message.content[0].text)


# ---------------------------------------------------------------------------
# 6. Self-Consistency Aggregation
# ---------------------------------------------------------------------------

def self_consistency(
    run_results: list[dict],
    normalize: bool = False,
) -> dict:
    """
    Aggregate target predictions across K independent LLM runs.

    Args:
        run_results: List of dicts, each with 'targets' and 'rationale'.
        normalize:   If True, express counts as fractions of K.

    Returns:
        Dict mapping protein name → {'count': int|float, 'rationales': list[str]}.
    """
    aggregated: dict[str, dict] = defaultdict(lambda: {"count": 0, "rationales": []})

    for result in run_results:
        for target in result.get("targets", []):
            aggregated[target]["count"] += 1
            rationale = result.get("rationale", {}).get(target, "")
            if rationale:
                aggregated[target]["rationales"].append(rationale)

    if normalize and run_results:
        k = len(run_results)
        for target in aggregated:
            aggregated[target]["count"] /= k

    return dict(aggregated)


def get_top_targets(aggregated: dict, min_count: int = 2) -> list[str]:
    """
    Filter to targets that appear in at least `min_count` runs, sorted by frequency.

    Args:
        aggregated: Output of self_consistency().
        min_count:  Minimum number of runs a target must appear in.

    Returns:
        List of target names sorted descending by count.
    """
    filtered = {k: v for k, v in aggregated.items() if v["count"] >= min_count}
    return sorted(filtered, key=lambda t: filtered[t]["count"], reverse=True)


# ---------------------------------------------------------------------------
# 7. Full Pipeline
# ---------------------------------------------------------------------------

def run_pipeline(
    drug_name: str,
    smiles: str,
    inchi_key: str,
    protein_list: list[str],
    few_shot_examples: list[dict],
    client: anthropic.Anthropic,
    k_runs: int = 5,
    pubmed_k: int = 10,
    pubmed_mail: Optional[str] = None,
    min_consensus_count: int = 2,
    max_targets_per_run: int = 5,
    sleep_between_runs: float = 0.5,
    model: str = "claude-sonnet-4-20250514",
    verbose: bool = True,
) -> dict:
    """
    End-to-end drug target detection pipeline.

    Pipeline stages (matching the architecture diagram):
      1. Fetch PubMed abstracts for grounding context.
      2. LLM generates a drug description from name + SMILES + InChI + abstracts.
      3. LLM refines the description.
      4. LLM predicts targets (few-shot) from candidate protein list — repeated K times.
      5. Self-consistency aggregation → consensus target list.

    Args:
        drug_name:           Name of the drug.
        smiles:              SMILES string.
        inchi_key:           InChI Key string.
        protein_list:        Candidate proteins the LLM must select from
                             (e.g. from STRING or a curated set).
        few_shot_examples:   List of dicts with keys:
                               - drug_name (str)
                               - description (str)
                               - targets (list[str])
        client:              Anthropic API client.
        k_runs:              Number of independent LLM inference runs (self-consistency).
        pubmed_k:            Number of PubMed abstracts to retrieve.
        pubmed_mail:         Email for NCBI Entrez API.
        min_consensus_count: Minimum run appearances for a target to be accepted.
        max_targets_per_run: Maximum targets the LLM may select per run.
        sleep_between_runs:  Seconds to wait between LLM calls (rate limiting).
        model:               Claude model identifier.
        verbose:             Print progress messages.

    Returns:
        Dictionary with full pipeline outputs:
          {
            "abstracts":            list[str],
            "drug_description":     str,
            "refined_description":  str,
            "all_run_results":      list[dict],
            "aggregated":           dict,
            "final_targets":        list[str],
          }
    """
    def log(msg):
        if verbose:
            print(f"[pipeline] {msg}")

    # ------------------------------------------------------------------
    # Stage 1: Fetch PubMed abstracts
    # ------------------------------------------------------------------
    log(f"Fetching up to {pubmed_k} PubMed abstracts for '{drug_name}' …")
    abstracts = fetch_abstracts(drug_name, k=pubmed_k, mail=pubmed_mail)
    log(f"  → Retrieved {len(abstracts)} abstracts.")

    # ------------------------------------------------------------------
    # Stage 2: Generate drug description
    # ------------------------------------------------------------------
    log("Stage 2: Generating drug description via LLM …")
    drug_description = generate_drug_description(
        drug_name=drug_name,
        smiles=smiles,
        inchi_key=inchi_key,
        abstracts=abstracts,
        client=client,
        model=model,
    )
    log("  → Description generated.")

    # ------------------------------------------------------------------
    # Stage 3: Refine description
    # ------------------------------------------------------------------
    log("Stage 3: Refining description via LLM …")
    refined_description = refine_drug_description(
        description=drug_description,
        client=client,
        model=model,
    )
    log("  → Description refined.")

    # ------------------------------------------------------------------
    # Stage 4: K-run target detection (few-shot, self-consistency)
    # ------------------------------------------------------------------
    log(f"Stage 4: Running {k_runs} independent target detection runs …")
    all_run_results = []
    for run_idx in tqdm(range(k_runs), desc="Target detection runs", disable=not verbose):
        result = detect_targets_single_run(
            refined_description=refined_description,
            protein_list=protein_list,
            few_shot_examples=few_shot_examples,
            client=client,
            model=model,
            max_targets=max_targets_per_run,
        )
        all_run_results.append(result)
        time.sleep(sleep_between_runs)

    # ------------------------------------------------------------------
    # Stage 5: Aggregate and filter
    # ------------------------------------------------------------------
    log("Stage 5: Aggregating results …")
    aggregated = self_consistency(all_run_results)
    final_targets = get_top_targets(aggregated, min_count=min_consensus_count)

    log(f"  → Final targets (≥{min_consensus_count} runs): {final_targets}")

    return {
        "abstracts": abstracts,
        "drug_description": drug_description,
        "refined_description": refined_description,
        "all_run_results": all_run_results,
        "aggregated": aggregated,
        "final_targets": final_targets,
    }

In [ ]:
"""
Example: Detect protein targets for Docetaxel
=============================================
Run this script after setting your ANTHROPIC_API_KEY environment variable:

    export ANTHROPIC_API_KEY="sk-ant-..."
    python example_usage.py
"""

import os
import json
import anthropic
from drug_target_pipeline import run_pipeline, fetch_string_proteins

# ---------------------------------------------------------------------------
# 0. Client
# ---------------------------------------------------------------------------
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# ---------------------------------------------------------------------------
# 1. Drug of interest
# ---------------------------------------------------------------------------
drug_name = "Docetaxel"
smiles    = "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3OC(=O)c5ccccc5)(CO4)OC(=O)[C@@H](N6C(=O)c7ccccc7C6=O)c8ccccc8)OC(=O)[C@@H](NC(=O)OC(C)(C)C)c9ccccc9)O2)O[C@@H]1O"
inchi_key = "ZDZOTLJHXYCWBA-VCVYQWHSSA-N"

# ---------------------------------------------------------------------------
# 2. Candidate protein list
# ---------------------------------------------------------------------------
# Option A: Fetch from STRING using the known primary target
print("Fetching candidate proteins from STRING …")
protein_list = fetch_string_proteins("TUBB", species=9606, limit=60, score_threshold=700)
print(f"  → {len(protein_list)} candidate proteins retrieved: {protein_list[:10]} …\n")

# Option B: Provide your own curated list
# protein_list = ["TUBB", "TUBA1A", "TP53", "BCL2", "CDK1", "AURKA", ...]

# ---------------------------------------------------------------------------
# 3. Few-shot examples (drug_name, short description, known targets)
# ---------------------------------------------------------------------------
# These guide the LLM on the format and level of specificity expected.
few_shot_examples = [
    {
        "drug_name": "Paclitaxel",
        "description": (
            "Paclitaxel is a taxane that stabilises microtubules by binding to the "
            "beta-tubulin subunit, preventing depolymerisation and arresting cells "
            "in the G2/M phase. It also activates BCL2 phosphorylation."
        ),
        "targets": ["TUBB", "BCL2"],
    },
    {
        "drug_name": "Imatinib",
        "description": (
            "Imatinib is a tyrosine kinase inhibitor that competitively binds the "
            "ATP-binding site of BCR-ABL1, c-KIT (CD117), and PDGFR-alpha/beta, "
            "blocking downstream proliferation signalling."
        ),
        "targets": ["ABL1", "KIT", "PDGFRA", "PDGFRB"],
    },
    {
        "drug_name": "Tamoxifen",
        "description": (
            "Tamoxifen is a selective oestrogen receptor modulator (SERM) that "
            "competitively binds ESR1 and ESR2, blocking oestrogen-driven "
            "transcription of proliferative genes in breast tissue."
        ),
        "targets": ["ESR1", "ESR2"],
    },
]

# ---------------------------------------------------------------------------
# 4. Run pipeline
# ---------------------------------------------------------------------------
results = run_pipeline(
    drug_name=drug_name,
    smiles=smiles,
    inchi_key=inchi_key,
    protein_list=protein_list,
    few_shot_examples=few_shot_examples,
    client=client,
    k_runs=5,                   # Number of self-consistency runs
    pubmed_k=10,                # PubMed abstracts to retrieve
    pubmed_mail=None,           # Set your email: "you@example.com"
    min_consensus_count=2,      # Must appear in ≥2 runs to be a final target
    max_targets_per_run=5,      # LLM selects at most 5 targets per run
    sleep_between_runs=0.5,     # Polite rate limiting
    verbose=True,
)

# ---------------------------------------------------------------------------
# 5. Results
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("FINAL TARGETS")
print("=" * 60)
for target in results["final_targets"]:
    count = results["aggregated"][target]["count"]
    rationale = results["aggregated"][target]["rationales"][0]
    print(f"  {target:20s}  (selected in {count}/{5} runs)")
    print(f"    Rationale: {rationale}\n")

print("\n" + "=" * 60)
print("FULL AGGREGATED VOTE (all proteins mentioned)")
print("=" * 60)
sorted_all = sorted(
    results["aggregated"].items(),
    key=lambda x: x[1]["count"],
    reverse=True,
)
for protein, data in sorted_all:
    print(f"  {protein:20s}  count={data['count']}")

# Optionally save everything
with open("pipeline_results.json", "w") as f:
    # Exclude heavy text fields to keep file readable
    summary = {
        "drug_name": drug_name,
        "final_targets": results["final_targets"],
        "aggregated": results["aggregated"],
        "refined_description": results["refined_description"],
    }
    json.dump(summary, f, indent=2)
print("\nFull results saved to pipeline_results.json")